In [85]:
import fastmcp
from fastmcp import FastMCP
# fastmcp for creating and connecting MCP servers,clients, and hosts

In [86]:
from langchain_core.tools import tool

#### tools in langchain

In [87]:
@tool
def multiply(a:int,b:int)->int:
    """Multiply two number"""
    return a*b
print('='*80)
print("the names of the tools is:",multiply.name)
print("the descriptions of the tools:",multiply.description)
print("the arguments of the tools",multiply.args)
print('='*80)
print(f"product:{multiply.invoke({'a':2,'b':3})}")
print('='*80)

the names of the tools is: multiply
the descriptions of the tools: Multiply two number
the arguments of the tools {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
product:6


#### Creating a calculator MCP Server

In [88]:
# creating fastmcp server object
mcp=FastMCP(
    name="CalculatorMCPServer",
    instructions="""
    This server provides data analysis tools
    call get_average() to analyze numerical data
    """
)
print("mcp object",mcp)

mcp object FastMCP('CalculatorMCPServer')


#### tools
* we define the MCP tools `add` and `subtrct`
* these will be called on the MCP object

In [89]:
@mcp.tool
def add(a:int,b:int)->int:
    """Add two integers together
    Args:
    a (int): the first integers
    b (int): the second integers
    
    Returns:
    int: the sum of a and b
    
    Example:
    >>> add(3,5)
    8
    """
    return a+b

@mcp.tool
def subtract(a:int,b:int)-> int:
    """Subtract the two given numbers
    
    Args:
    a (int): first integer 
    b (int): the second integer
    
    Return:
    int : the sum of `a` and `b`.
    
    Example:
    >>> subtract(3,1)
    1
    """
    return a-b

#### Resources

In [90]:
@mcp.resource('file:///endpoint/{name}')
def return_template_document(name:str)->str:
    """Read a document by name"""
    with open(f"resources_path/{name}",'r') as f:
        return f.read()

In [91]:
import os 
def make_dirs():
    if os.path.exists("resources_path"):
        print("path directory already exist")
    else:
        os.makedirs("resources_path")
        print("path directory created")

In [92]:
make_dirs()

path directory already exist


When you call this resources, it will require the input `{name}` to identify which document to retrieve.`path/{name}` is where files actually exist on disk. The function reads from this physical file system location to return the content.

**Note:** The URI endpoint is the MCP address for requesting resources, while the path is the actual storage location on your system. They are related but distinct.


In [93]:
@mcp.resource("file://endpoint2/{name}")
def read_document(name:str)->str:
    """Read a document by name from the path directory"""
    try:
        with open(f"resources_path/{name}","r") as f:
            return f.read()
    except FileNotFoundError:
        return f"document {name} not found in the path directory"
    except Exception as e:
        return f"Error reading document:{str(e)}"
    

#### Prompts 

Prompts are consistent, reusable templates that can be called for simple, repetitive tasks. They capture domain expertise in a structured way, so instead of reinventing instructions each time, the AI can rely on a proven pattern.


In [94]:
@mcp.prompt(title="Code Review")
def review_code(code:str)->str:
    return f"please review this code:\n\n{code}"

#### Creating a Client: In-memory-transport

In [95]:
from fastmcp import Client
client=Client(mcp)
print(f"clinet:{client}")

clinet:<fastmcp.client.client.Client object at 0x00000211566F5950>


#### Creating tools

In [96]:
async def call_add_tool(a:int,b:int):
    async with client:
        result=await client.call_tool("add",{"a":a,"b":b})
        return result

In [97]:
response=await call_add_tool(4,5)
print(response.content)

[TextContent(type='text', text='9', annotations=None, meta=None)]


In [98]:
# The actual answer/data
print("\nResult Data .data :")
print(response.data)  # 9

# Content (text format)
print("\nContent (as text):")
print(response.content[0].text)  # "9"

# Structured content (as dictionary)
print("\nStructured Content:")
print(response.structured_content)  


Result Data .data :
9

Content (as text):
9

Structured Content:
{'result': 9}


#### Fetching Available Tools

In [99]:
async with client:
    tools = await client.list_tools()
    print('='*80)
    print("Availabe tools:")
    for tool in tools:
        print('='*80)
        print("the name of tool is:",tool.name)
        print("the description of the tool is:",tool.description)

Availabe tools:
the name of tool is: add
the description of the tool is: Add two integers together
Args:
a (int): the first integers
b (int): the second integers

Returns:
int: the sum of a and b

Example:
>>> add(3,5)
8
the name of tool is: subtract
the description of the tool is: Subtract the two given numbers

Args:
a (int): first integer 
b (int): the second integer

Return:
int : the sum of `a` and `b`.

Example:
>>> subtract(3,1)
1


#### Fetching the resources

In [100]:
async def call_resources(name):
    async with client:
        result= await client.read_resource(f"file:///endpoint/{name}")
        return result
    

In [108]:
response=await call_resources("example.txt")
resource=response[0]


In [109]:
print(f"uri:      {resource.uri}")
print(f"mimeType: {resource.mimeType}")
print(f"meta:     {resource.meta}")
print(f"text:     {resource.text}")

uri:      file:///endpoint/example.txt
mimeType: text/plain
meta:     None
text:     hi, welcome to the mcp tutorial

this is introuductions about mcp 


In [103]:
async def call_resource_2(name):
    async with client:
        result= await client.read_resource(f"file://endpoint2/{name}")
        return result

In [104]:
response=await call_resource_2('readme.txt')
resource=response[0]

In [105]:
print(f"uri:      {resource.uri}")
print(f"mimeType: {resource.mimeType}")
print(f"meta:     {resource.meta}")
print(f"text:     {resource.text}")

uri:      file://endpoint2/readme.txt
mimeType: text/plain
meta:     None
text:     # Documents Folder

This folder contains documents accessible through the Calculator MCP Server.

## Available Documents:

- examples.txt - Examples of how to use the calculator
- README.txt - This file



#### Prompts
we'll also create a function to call the prompts method to review code. the only parameters is the code content to be reviewed

In [113]:
async def call_prompt(code):
    async with client:
        result=await client.get_prompt("review_code",{"code":code})
        return result

In [114]:
response = await call_prompt("CODE TO BE REVIEWED")

In [115]:
message=response.messages[0]
print(f"Prompt Role:{message.role}")
print(f"Prompt Content:{message.content.text}")

Prompt Role:user
Prompt Content:please review this code:

CODE TO BE REVIEWED
